<a href="https://colab.research.google.com/github/votaquangnhat/LifeMate-Baseline/blob/main/notebooks/lifemate_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LifeMate — GitHub → Colab → public submission
Choose **Runtime → Change runtime type → GPU**, then run the cells in order.
This notebook uses Colab's Python runtime and its installed CUDA PyTorch. No virtual environment or Drive mount is used.
The dataset is downloaded with gdown. Results are saved under `/content/lifemate_runs`; download them before the runtime is deleted.


In [1]:
# 1. Clone the repository and use its project directory.
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/votaquangnhat/LifeMate-Baseline.git"
PROJECT = Path("/content/lifemate")
if not PROJECT.exists():
    !git clone {REPO_URL} {PROJECT}
    assert _exit_code == 0, "Git clone failed. Check repository access."
else:
    print("Using existing clone:", PROJECT)
os.chdir(PROJECT)
assert (PROJECT / "requirements-colab.txt").is_file(), "Push the updated code, including requirements-colab.txt, to GitHub first."
!nvidia-smi
assert _exit_code == 0, "Select a GPU runtime first."


Cloning into '/content/lifemate'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 59 (delta 14), reused 53 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 48.60 KiB | 12.15 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Fri Nov  6 12:30:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+==

In [2]:
# 2. Download dataset
!pip -q install gdown
!gdown 1f5Vdnm3fhLdBYZFmRPPYvKscmFmbozmQ -O /content/train+public_test.zip

from pathlib import Path
ARCHIVE = Path("/content/train+public_test.zip")
assert _exit_code == 0 and ARCHIVE.is_file(), "Dataset download failed; check the Drive file's access/quota."


Downloading...
From (original): https://drive.google.com/uc?id=1f5Vdnm3fhLdBYZFmRPPYvKscmFmbozmQ
From (redirected): https://drive.google.com/uc?id=1f5Vdnm3fhLdBYZFmRPPYvKscmFmbozmQ&confirm=t&uuid=c2d26f3a-f7cb-4427-b684-f72e40dcc749
To: /content/train+public_test.zip
100% 876M/876M [00:04<00:00, 188MB/s]


In [3]:
# 3. Extract the dataset on Colab's local disk.
import shutil
DATA_DIR = Path("/content/lifemate-data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not list(DATA_DIR.glob("**/public_test/public_test.jsonl")):
    shutil.unpack_archive(str(ARCHIVE), str(DATA_DIR))
manifest_paths = list(DATA_DIR.glob("**/public_test/public_test.jsonl"))
assert len(manifest_paths) == 1, f"Expected one public_test manifest; found {manifest_paths}"
DATA_ROOT = manifest_paths[0].parent.parent
assert (DATA_ROOT / "train/train.jsonl").is_file(), "The ZIP must contain train and public_test."
print("Data root:", DATA_ROOT)


Data root: /content/lifemate-data/train+public_test


In [4]:
# 4. Install packages into the CURRENT Colab runtime. Keep Colab's CUDA PyTorch.
!{sys.executable} -m pip install -q -r requirements-colab.txt
assert _exit_code == 0, "Dependency installation failed. Inspect pip's error above."

import torch
from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "This pipeline requires PyTorch >=2.6; use an up-to-date Colab runtime."
assert torch.cuda.is_available(), "Select a GPU runtime."
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
# If Colab explicitly requests a session restart, restart, then rerun cells 1 onward.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.0 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.0 which is incompatible.
diffusers 0.42.1 requires safetensors>=0.8.0, but you have safetensors 0.6.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.0 which is incompatible.
Python: /usr/bin/python3
PyTorch: 2.11.0+cu130
GPU: NVIDIA L4


In [5]:
# 5. Download the baseline model weights automatically.
!{sys.executable} scripts/download_models.py --profile baseline
assert _exit_code == 0, "Model download failed. Fix it and rerun this cell."


Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]
model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 32.8kB [00:00, 1.92MB/s]

README.md: 8.17kB [00:00, 3.49MB/s]

config.json: 100% 727/727 [00:00<00:00, 5.44MB/s]

generation_config.json: 100% 238/238 [00:00<00:00, 2.21MB/s]
merges.txt: 1.67MB [00:00, 60.2MB/s]

model-00001-of-00003.safetensors:   0% 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0% 0.00/99.6M [00:00<?, ?B/s]


model-00002-of-00003.safetensors:   0% 0.00/3.99G [00:00<?, ?B/s]



vocab.json: 0.00B [00:00, ?B/s]




tokenizer_config.json: 9.38kB [00:00, 35.2MB/s]
vocab.json: 2.78MB [00:00, 113MB/s]




tokenizer.json:   0% 0.00/11.4M [00:00<?, ?B/s]



tokenizer.json: 100% 11.4M/11.4M [00:01<00:00, 9.04MB/s]


model-00003-of-00003.safetensors: 100% 99.6M/99.6M [00:01<00:00, 63.0MB/s]


model-00003-of-00003.safetensors: 100% 99.6M/99.6M [00:01<00:00, 55.3MB/s]



model-00002-of-00003.safetensors:   7% 268M/3.99G [00:02<00:22, 166

In [6]:
# 6. Configure dataset paths and prepare training-only retrieval examples.
import json
cfg = json.loads((PROJECT / "configs/colab.json").read_text(encoding="utf-8-sig"))
cfg["data_root"] = str(DATA_ROOT)
CONFIG = PROJECT / "configs/colab-runtime.json"
CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
!{sys.executable} -m baseline.prepare --data-root "{DATA_ROOT}"
assert _exit_code == 0, "Data preparation failed."
!{sys.executable} -m baseline.doctor --config "{CONFIG}" --split public_test
assert _exit_code == 0, "Preflight failed."


Pool=1345, dev=200, SFT=1345, skipped_long=0 -> artifacts
Python 3.13.16 | executable: /usr/bin/python3
torch: 2.11.0+cu130
transformers: 4.57.1
peft: 0.17.1
soundfile: 0.13.1
scikit-learn: 1.7.2
public_test: 634 samples; catalogs consistent
CUDA: True
GPU: NVIDIA L4
llm: models/Qwen3-4B-Instruct-2507; adapter=None
asr: models/PhoWhisper-small; adapter=None
Preflight passed. This does not certify accuracy, GPU memory fit, or the 30s limit.


In [7]:
# 7. Test ONE training audio before running the full public split.
from datetime import datetime, timezone
RUN_DIR = Path("/content/lifemate_runs") / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
RUN_DIR.mkdir(parents=True, exist_ok=False)
SMOKE = RUN_DIR / "colab_smoke.jsonl"
!{sys.executable} -m baseline.run --config "{CONFIG}" --split train --ids artifacts/splits/dev_ids.json --limit 1 --seconds 120 --output "{SMOKE}" --verbose
assert _exit_code == 0, "Smoke inference failed. Review the error/diagnostics."
!{sys.executable} -m baseline.inspect_run "{SMOKE}" --require-complete
assert _exit_code == 0, "The smoke run did not finish normally."
print("Output directory:", RUN_DIR)


Loading planner: models/Qwen3-4B-Instruct-2507 on cuda:0 ...
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793968343.836077    1560 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793968343.906105    1560 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793968346.119239    1560 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:

## Full public run
Continue after the smoke run completes. The next cell runs every public sample with the 120-second cooperative budget.
Results are saved locally under `/content/lifemate_runs`. Download the completed result in cell 10 before the runtime is deleted.
A runtime disconnect may leave a partial file; full-split validation must pass before submission. There is no automatic resume.


In [8]:
# 8. Generate ALL public_test predictions. No sample limit here.
SUBMISSION = RUN_DIR / "submission.jsonl"
!{sys.executable} -m baseline.run --config "{CONFIG}" --split public_test --seconds 120 --output "{SUBMISSION}"
assert _exit_code == 0, "Inference reported failures. Inspect diagnostics before submitting."


Loading planner: models/Qwen3-4B-Instruct-2507 on cuda:0 ...
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793968376.320871    1783 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793968376.393409    1783 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793968378.841113    1783 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:

In [9]:
# 9. Validate the entire submission and review timeouts/errors.
!{sys.executable} -m baseline.validate_submission "{SUBMISSION}" --data-root "{DATA_ROOT}" --split public_test
assert _exit_code == 0, "Submission validation failed. Do not submit a partial/invalid file."
!{sys.executable} -m baseline.inspect_run "{SUBMISSION}"
assert _exit_code == 0, "Could not inspect the run."
print("Submit THIS file:", SUBMISSION)


Validated 634 samples: coverage and schemas OK. Hash values are not replay-verified.
Samples: 634 | stop reasons: {'finished': 599, 'asked_missing_info': 35}
Above 30 seconds: 28
Unsuccessful tool outcomes: 27
Completion does not prove that the model selected the correct actions.
Submit THIS file: /content/lifemate_runs/20261106_123213_087239_UTC/submission.jsonl


In [10]:
# 10. Download submission.jsonl, then upload it to the competition portal.
from google.colab import files
files.download(str(SUBMISSION))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Later: teammate checkpoints
Download/upload the checkpoints into Colab's local disk and change `cfg["asr"]["path"]` and `cfg["llm"]["adapter"]` in cell 6.
See `TEAM_HANDOFF.md` for checkpoint formats. Use a new output directory for each experiment.

If a cell fails, fix it before continuing. To start another experiment, rerun cell 7 to create a new result directory.
If Colab requests a restart after installation, restart the session and rerun cells 1 onward. The runtime's Python executes all commands; no separate virtual environment is created.
The Colab CUDA PyTorch version is retained; GPU performance must be measured on the actual runtime.
